In [ ]:
import sys
import torch
import functools
import matplotlib.pyplot as plt
import argparse, yaml, os
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.transforms as transforms
import seaborn as sns
import pandas as pd
import tqdm
import glob

from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from types import SimpleNamespace

from IPython.display import clear_output

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params as model_params_tm
from texture_prior.params import statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")
import DistanceMemoryModel
import encoders

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir


sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")
device = 'cuda'
# get soem textures
pc_dims = 256

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

texture_model = encoders.AudioTextureEncoder(
    statistics_dict=statistics_set.statistics,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    coch_filter=48,
    device=device
)

def compute_likelihood(score_model, input_stats, ckpt):
    """Computing the actual "prior" value
    If the score function can be treated as the vector field govering the temporal evolution
    of $x_t$ then we can integrate the ODE and apply a change of basis to evaluate the prior
    """
    score_model.load_state_dict(torch.load(ckpt))
    input_stats = input_stats.to(device)

    _, bpd = ode_likelihood(input_stats, score_model, marginal_prob_std_fn,
                            diffusion_coeff_fn,
                            input_stats.shape[0], device=device, eps=1e-5)
    
    return bpd

def parse(d):
  x = SimpleNamespace()
  _ = [setattr(x, k, parse(v)) if isinstance(v, dict) else setattr(x, k, v) for k, v in d.items() ]    
  return x

In [ ]:
sounds_list[0]

In [ ]:
texture_model(sounds_list[0]).shape

In [ ]:
pc_texture_model(sounds_list[0]).shape

In [ ]:
# get soem textures
pc_dims = 2

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)


all_sounds_pc = []
for sound in sounds_list:

    x = pc_texture_model(sound).detach().to('cpu').numpy()
    all_sounds_pc.append(x)
    plt.scatter(x=x[0], y=x[1], color='b', alpha=0.4)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Textures used in memory experiments projected to PC space")

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist

In [ ]:
plt.hist(pdist(all_sounds_pc), bins=100, color='b', alpha=0.5);
plt.title("Pairwise distances of textures with first two PCs");

In [ ]:
# get soem textures
pc_dims = 4096

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)


all_sounds_pc_higher = []
for sound in sounds_list:

    x = pc_texture_model(sound).detach().to('cpu').numpy()
    all_sounds_pc_higher.append(x)
    #plt.scatter(x=x[0], y=x[1], color='b', alpha=0.4)

plt.hist(pdist(all_sounds_pc_higher), bins=100, color='b', alpha=0.5);
plt.title(f"Pairwise distances of textures with first {pc_dims} PCs");

In [ ]:
# get soem textures
pc_dims = 10325

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)


all_sounds_pc_higher = []
for sound in sounds_list:

    x = pc_texture_model(sound).detach().to('cpu').numpy()
    all_sounds_pc_higher.append(x)
    #plt.scatter(x=x[0], y=x[1], color='b', alpha=0.4)

plt.hist(pdist(all_sounds_pc_higher), bins=100, color='b', alpha=0.5);
plt.title(f"Pairwise distances of textures with first {pc_dims} PCs");

In [ ]:
# get soem textures


texture_model = encoders.AudioTextureEncoder(
    statistics_dict=statistics_set.statistics,
    model_params=model_params_tm,
    sr=20000,
    coch_filter=48,
    rms_level=0.05,
    duration=2.0,
    device=device
)


all_sounds_full = []
for sound in sounds_list:

    x = texture_model(sound).detach().to('cpu').numpy()
    all_sounds_full.append(x)
    #plt.scatter(x=x[0], y=x[1], color='b', alpha=0.4)

plt.hist(pdist(all_sounds_full), bins=100, color='b', alpha=0.5);
plt.title(f"Pairwise distances of textures (full ambient)");

In [ ]:
plt.hist(pdist(all_sounds_full), bins=100, color='r', alpha=0.5, label="Full ambient");
plt.hist(pdist(all_sounds_pc_higher), bins=100, color='b', alpha=0.5, label=f"{pc_dims} PCs");
plt.legend()
plt.title(f"Pairwise distances of textures");